# OpenAI Agents SDK: From Fundamentals to Production-Ready AI Agents

## Notebook 1.2 — Introduction to Modern AI Agents  
### From Rule-Based Software to LLM-Powered Agents

---

**Prerequisite:** Notebook 1.1 — Parts 1 and 2

> **Central question**
>
> What changed between classical AI agents and the modern AI agents built with large language models?

Modern AI agents did not appear suddenly. They emerged from decades of progress in:

- symbolic AI;
- expert systems;
- machine learning;
- deep learning;
- foundation models;
- large language models;
- tool calling;
- and scalable software infrastructure.

This notebook builds that story carefully. By the end, you should be able to explain not only **what** a modern AI agent is, but also **why** such agents became practical only recently.

---

### How to use this notebook

1. Read the explanatory cells in order.
2. Predict every code example before running it.
3. Pause at the knowledge checks.
4. Use the discussion prompts during classroom teaching.
5. Complete the design activity before moving to Part 2.

>
> This notebook deliberately avoids the OpenAI Agents SDK API. Students first need a clear mental model of modern agents before learning SDK syntax.

## 1. Where We Are in the Course

Notebook 1.1 established the classical foundation.

```text
Artificial Intelligence
        │
        ├── Intelligence and rational behaviour
        ├── Agents and environments
        ├── Observations and actions
        ├── Goals and performance measures
        ├── Reflex agents
        ├── Model-based agents
        ├── Goal-based agents
        └── Utility-based agents
```

Notebook 1.2 now asks:

```text
How did these classical ideas evolve into modern LLM-powered agents?
```

The transition can be visualised as:

```text
Classical Agent Theory
        │
        ▼
Large Language Models
        │
        ▼
Reasoning + Tools + Memory + Control Loop
        │
        ▼
Modern AI Agent
```

A modern AI agent still follows the same basic loop:

```text
Observe → Decide → Act → Observe Again
```

What has changed is the **decision-making engine** and the **range of actions** available.

## 2. Learning Objectives

By the end of this notebook, you should be able to:

1. explain why traditional software is insufficient for some real-world tasks;
2. describe the evolution from rule-based systems to modern AI agents;
3. distinguish symbolic AI, machine learning, deep learning, foundation models, and LLMs;
4. explain why language models created a new interface for software;
5. distinguish a model from an agent;
6. compare a chatbot, assistant, workflow, and AI agent;
7. identify the major characteristics of a modern AI agent;
8. analyse whether a system is genuinely agentic;
9. recognise where deterministic software remains preferable;
10. design a conceptual modern agent for a real-world problem.

## 3. Why Do We Need Modern AI Agents?

Many useful tasks are not single-step problems.

Consider the request:

> “Plan a three-day educational trip to Delhi for 25 students within a budget of £3,000.”

A useful system may need to:

1. understand the request;
2. identify missing information;
3. search for transport options;
4. search for accommodation;
5. estimate food and local travel costs;
6. compare several plans;
7. verify the budget;
8. ask the organiser for approval;
9. create an itinerary;
10. send the final document.

This is not simply:

```text
Input → Output
```

It is:

```text
Goal
  ↓
Understand the situation
  ↓
Break the goal into tasks
  ↓
Use external tools
  ↓
Inspect results
  ↓
Revise the plan
  ↓
Ask for approval
  ↓
Complete the task
```

Modern AI agents are designed for this kind of **multi-step, context-dependent work**.

## 4. The Limitation of Traditional Input–Output Programs

Traditional software is often written as an explicit mapping:

```text
Input + Programmed Rules → Output
```

For example:

```python
if balance >= withdrawal_amount:
    approve_withdrawal()
else:
    reject_withdrawal()
```

This approach is excellent when:

- rules are known;
- inputs are structured;
- possible conditions are limited;
- behaviour must be predictable;
- errors are expensive.

However, it becomes difficult when:

- users express the same intent in many ways;
- information is incomplete;
- tasks require interpretation;
- the correct next action depends on context;
- the environment changes;
- the number of possible situations is very large.

Traditional software is not obsolete. The important question is:

> Which parts of a system should be deterministic, and which parts require flexible reasoning?

## 5. Example: A Rule-Based Support System

Suppose we want to route customer messages.

A rule-based approach might use keywords:

```text
IF message contains “refund” → Refund Team
IF message contains “password” → Account Team
IF message contains “delivery” → Logistics Team
```

This looks reasonable until users write:

- “I want my money back.”
- “I can no longer access my account.”
- “My parcel still hasn’t arrived.”
- “The product was charged twice and I also cannot log in.”

The system now needs:

- synonyms;
- spelling variations;
- multiple intents;
- context;
- priority;
- and ambiguity handling.

The number of rules grows rapidly.

In [ ]:
# Example 1: A keyword-based support router

def route_support_message(message: str) -> str:
    """Route a support message using explicit keyword rules."""
    text = message.lower()

    if "refund" in text:
        return "Refund Team"
    elif "password" in text:
        return "Account Access Team"
    elif "delivery" in text:
        return "Logistics Team"
    else:
        return "General Support"


messages = [
    "I need a refund.",
    "I forgot my password.",
    "My delivery is late.",
    "I want my money back.",
    "I cannot access my account.",
    "My parcel has not arrived."
]

for message in messages:
    print(f"{message!r:40} → {route_support_message(message)}")

### Interpreting Example 1

The first three messages match the exact vocabulary expected by the programmer.

The next three express similar meanings using different words.

The program does not actually understand:

- “money back” as a refund;
- “cannot access my account” as an account problem;
- “parcel has not arrived” as a delivery problem.

It only performs string matching.

```text
Meaning intended by user
          ≠
Words explicitly recognised by program
```

A programmer can add more keywords, but this creates a maintenance problem.

> **Think like an AI engineer**
>
> Do not replace every rule with an LLM. Instead, use language models where semantic interpretation is valuable and deterministic rules where exact control is necessary.

## 6. Rule Explosion

A rule-based system may begin simply:

```text
3 intents × 3 phrases = 9 rules
```

But real systems may contain:

- 50 intents;
- 20 common phrases per intent;
- several languages;
- spelling errors;
- multiple intents in one message;
- exceptions;
- priority rules;
- policy constraints.

Even a simplified estimate gives:

```text
50 intents × 20 expressions × 5 variations = 5,000 patterns
```

The deeper problem is not merely the number of rules. It is the interaction among them.

```text
Rule A applies
Rule B also applies
Rule C overrides Rule A
Rule D applies only for premium customers
Rule E applies after 30 days
```

This is called **combinatorial complexity**: the number of possible combinations grows faster than the number of individual rules.

In [ ]:
# Example 2: Visualising rule growth

intents = [5, 10, 25, 50, 100]
phrases_per_intent = 20
variations_per_phrase = 5

print("Estimated rule patterns")
print("-" * 32)

for number_of_intents in intents:
    patterns = (
        number_of_intents
        * phrases_per_intent
        * variations_per_phrase
    )
    print(
        f"{number_of_intents:3} intents "
        f"→ approximately {patterns:6,} patterns"
    )

### What This Example Does—and Does Not—Show

The calculation is illustrative rather than a law of software engineering.

It demonstrates an important intuition:

> When language and context are involved, enumerating every valid expression becomes impractical.

Machine learning approached this problem differently:

```text
Rule-Based Approach:
Programmer writes the decision rules.

Machine-Learning Approach:
System learns decision boundaries from examples.
```

Large language models go further:

```text
LLM-Based Approach:
A general-purpose model interprets language and follows instructions
without requiring a separate training process for every narrow task.
```

## Knowledge Check 1

Answer before continuing.

1. Why is traditional software still valuable?
2. What kinds of tasks are difficult to solve using only explicit rules?
3. What does “rule explosion” mean?
4. Why is keyword matching different from understanding meaning?
5. Give one example where deterministic code is safer than an LLM.
6. Give one example where semantic interpretation is more useful than fixed rules.
7. In a customer-support system, which components might remain deterministic?

## 7. The Evolution of Intelligent Software

Modern AI agents combine ideas from several generations of computing.

```text
1950s–1960s    Symbolic AI
       ↓
1960s–1980s    Expert Systems
       ↓
1990s–2010s    Statistical Machine Learning
       ↓
2010s           Deep Learning
       ↓
Late 2010s      Foundation Models
       ↓
2020s           Large Language Models
       ↓
Current Era     LLM-Powered Agents
```

Each generation solved some limitations of the previous one, while introducing new challenges.

The history is not a clean replacement sequence. Modern systems still use:

- rules;
- databases;
- optimisation;
- machine learning;
- deep learning;
- language models;
- and classical software engineering.

A production agent is usually a **hybrid system**.

## 8. Symbolic AI

Symbolic AI represents knowledge explicitly using symbols and rules.

Example:

```text
Human(Socrates)
∀x Human(x) → Mortal(x)
Therefore: Mortal(Socrates)
```

In software form:

```text
IF entity is human
THEN entity is mortal
```

### Strengths

- rules are visible;
- reasoning can be inspected;
- conclusions may be logically justified;
- behaviour is predictable within the encoded world.

### Limitations

- knowledge must be entered manually;
- real-world concepts are difficult to encode completely;
- uncertainty is difficult to manage;
- language is ambiguous;
- exceptions accumulate.

Symbolic AI remains useful in:

- compliance rules;
- configuration systems;
- formal verification;
- tax calculation;
- and knowledge graphs.

## 9. Expert Systems

An expert system attempts to reproduce the decision rules of a human specialist.

A simplified architecture is:

```text
┌───────────────────────┐
│   Knowledge Base      │
│ facts + expert rules  │
└───────────┬───────────┘
            │
            ▼
┌───────────────────────┐
│   Inference Engine    │
│ applies the rules     │
└───────────┬───────────┘
            │
            ▼
┌───────────────────────┐
│ Recommendation/Result │
└───────────────────────┘
```

Example medical rule:

```text
IF fever = high
AND cough = present
AND oxygen_level < threshold
THEN recommend urgent assessment
```

Expert systems demonstrated that computers could support complex decisions. But they were expensive to build because experts and programmers had to encode knowledge manually.

In [ ]:
# Example 3: A tiny expert system

def equipment_diagnosis(
    powers_on: bool,
    unusual_sound: bool,
    overheats: bool
) -> list[str]:
    """Return possible diagnoses using explicit expert rules."""

    diagnoses = []

    if not powers_on:
        diagnoses.append("Check power supply and connection.")

    if powers_on and unusual_sound:
        diagnoses.append("Inspect moving components.")

    if powers_on and overheats:
        diagnoses.append("Inspect cooling system.")

    if powers_on and unusual_sound and overheats:
        diagnoses.append("Stop operation and request specialist inspection.")

    if not diagnoses:
        diagnoses.append("No known issue matched the rule base.")

    return diagnoses


cases = [
    {"powers_on": False, "unusual_sound": False, "overheats": False},
    {"powers_on": True, "unusual_sound": True, "overheats": False},
    {"powers_on": True, "unusual_sound": True, "overheats": True},
]

for index, case in enumerate(cases, start=1):
    print(f"Case {index}: {case}")
    for diagnosis in equipment_diagnosis(**case):
        print(" -", diagnosis)
    print()

### Interpreting the Expert-System Example

The program is easy to inspect.

We can point to the exact rule responsible for every recommendation.

However, it cannot handle a new observation such as:

> “The equipment smells burnt after running for ten minutes.”

Unless a programmer adds a rule for this situation, the system has no way to generalise from related cases.

This reveals a central limitation:

```text
Expert system knowledge
=
Only the knowledge explicitly encoded
```

By contrast, machine-learning systems can learn patterns from examples.

## 10. Machine Learning

Machine learning changes the source of the decision logic.

### Traditional programming

```text
Rules + Data → Answers
```

### Machine learning

```text
Data + Answers → Learned Model
```

The model can later make predictions:

```text
New Data + Learned Model → Prediction
```

Example:

Instead of writing every rule for spam email, we provide examples labelled:

- spam;
- not spam.

The learning algorithm identifies useful statistical patterns.

### What machine learning improved

- reduced dependence on hand-written rules;
- supported noisy data;
- enabled pattern recognition;
- improved with additional examples.

### What it still required

- task-specific datasets;
- training;
- feature design in many early systems;
- separate models for separate tasks.

## 11. Deep Learning

Deep learning uses multi-layer neural networks to learn increasingly useful representations.

A simplified view is:

```text
Raw Input
   ↓
Low-Level Patterns
   ↓
Intermediate Features
   ↓
High-Level Representation
   ↓
Prediction
```

For an image:

```text
pixels → edges → textures → shapes → object
```

For language:

```text
tokens → phrases → relationships → contextual representation
```

Deep learning enabled major advances in:

- computer vision;
- speech recognition;
- machine translation;
- recommendation;
- and natural-language processing.

Its major contribution was not only prediction. It reduced the need for humans to manually define every useful feature.

## 12. Foundation Models

Earlier machine-learning models were often built for one narrow task.

```text
Model A → Spam detection
Model B → Sentiment analysis
Model C → Translation
Model D → Question classification
```

A **foundation model** is trained broadly and can be adapted to many downstream tasks.

```text
Large-scale pretraining
        │
        ├── summarisation
        ├── classification
        ├── translation
        ├── question answering
        ├── code generation
        └── many other tasks
```

The word “foundation” suggests that many applications can be built on top of the same underlying model.

This changed software development because developers no longer needed to train a separate model for every language task.

## 13. Large Language Models

A large language model, or LLM, is a foundation model trained on large quantities of language and related data.

At a simplified level, an LLM predicts the next token.

```text
Input tokens
    ↓
Contextual processing
    ↓
Probability distribution over next token
    ↓
Selected token
    ↓
Repeat
```

Example:

```text
“The capital of France is …”
```

The model assigns a high probability to a token representing “Paris”.

However, modern LLM behaviour is richer than simple memorisation because the model learns broad statistical relationships among:

- concepts;
- facts;
- grammar;
- styles;
- instructions;
- code;
- and patterns of reasoning.

> **Important distinction**
>
> Next-token prediction is the training mechanism. The resulting model can still support complex behaviours such as explanation, transformation, planning, classification, and code generation.

In [ ]:
# Example 4: A tiny next-token demonstration

# This is NOT a real language model.
# It is a simple frequency table used to explain the idea.

training_sentences = [
    "agents use tools",
    "agents use memory",
    "agents use instructions",
    "students use notebooks",
    "developers use tools",
]

next_word_counts = {}

for sentence in training_sentences:
    words = sentence.split()
    for current, following in zip(words, words[1:]):
        next_word_counts.setdefault(current, {})
        next_word_counts[current][following] = (
            next_word_counts[current].get(following, 0) + 1
        )

for word, possible_next_words in next_word_counts.items():
    print(f"After {word!r}: {possible_next_words}")

### What the Tiny Demonstration Teaches

The example records which words followed other words in a tiny dataset.

A real LLM is vastly more sophisticated:

- it works with tokens rather than ordinary words;
- it represents context using high-dimensional vectors;
- it uses many neural-network layers;
- it considers relationships across long passages;
- it learns from enormous datasets;
- it generates probabilities dynamically.

Still, the example provides a useful first intuition:

```text
Language generation
=
Repeatedly predicting plausible continuations
using context
```

### Common misconception

> “If an LLM predicts the next token, it cannot reason.”

This conclusion does not follow automatically.

A system trained to predict language may learn internal representations that support behaviours resembling reasoning. The practical engineering question is not whether the model “thinks like a human,” but:

> How reliably can it perform the required task under controlled conditions?

## Knowledge Check 2

Match each concept with its defining idea.

| Concept | Defining Idea |
|---|---|
| Symbolic AI | ? |
| Expert system | ? |
| Machine learning | ? |
| Deep learning | ? |
| Foundation model | ? |
| Large language model | ? |

Then answer:

1. Why were expert systems expensive to maintain?
2. What changed when machine learning replaced hand-written rules with learned patterns?
3. How did deep learning reduce manual feature engineering?
4. Why is a foundation model different from a narrow model?
5. Why is an LLM useful across many language tasks?
6. What is misleading about saying that an LLM “only predicts the next token”?

## 14. Why LLMs Changed the Software Interface

Traditional software expects structured commands.

Examples:

```text
SELECT customer WHERE id = 152
```

```text
POST /booking
{
  "destination": "Delhi",
  "travellers": 25
}
```

Humans naturally communicate differently:

> “Find a suitable educational trip for 25 students. Keep it within our budget and avoid overnight travel.”

LLMs provide a flexible interface between human language and software operations.

```text
Human Language
      ↓
Language Model
      ↓
Structured Intent
      ↓
Software Action
```

This is powerful because users no longer need to know:

- database syntax;
- API formats;
- command names;
- exact workflow steps;
- or programming languages.

## 15. Natural Language as a Universal Control Layer

Before LLMs, every software system required a specific interface:

- forms;
- menus;
- commands;
- buttons;
- query languages;
- dashboards.

With an LLM, natural language can become a general control layer.

```text
User:
“Compare the three latest proposals,
identify financial risks,
and draft questions for the meeting.”

              ↓

LLM interprets the request

              ↓

System selects files, analysis tools,
calculations, and document generation
```

This does not remove the need for software interfaces. It introduces a new interface capable of interpreting flexible instructions.

### Why this matters for agents

An agent receives goals in language, translates them into actions, and uses tools to affect external systems.

## 16. Generality: One Model, Many Tasks

A traditional system may require separate components:

```text
Summariser
Classifier
Translator
Code generator
Question-answering model
Intent detector
```

An LLM can often perform all of these tasks using different instructions.

Examples:

```text
“Summarise this report.”
“Classify this ticket.”
“Translate this message.”
“Write a Python function.”
“Extract the invoice number.”
“Explain this error.”
```

The model is not equally reliable at every task, but its generality is a major engineering advantage.

This makes it possible to build agents that encounter new combinations of tasks without pre-programming every sequence.

In [ ]:
# Example 5: One interface, multiple task types

def build_instruction(task: str, content: str) -> str:
    """Construct a clear instruction for different task types."""

    task_templates = {
        "summarise": (
            "Summarise the following content in three bullet points:\n"
        ),
        "classify": (
            "Classify the following message as Billing, Technical, "
            "Account, or Other:\n"
        ),
        "extract": (
            "Extract the person's name and date from the following text. "
            "Return a dictionary:\n"
        ),
        "rewrite": (
            "Rewrite the following text in a professional tone:\n"
        ),
    }

    if task not in task_templates:
        raise ValueError(f"Unsupported task: {task}")

    return task_templates[task] + content


examples = [
    ("summarise", "The programme trained 150 students across five domains."),
    ("classify", "I was charged twice for the same order."),
    ("extract", "Riya will attend the workshop on 14 August."),
    ("rewrite", "send me the file fast"),
]

for task, content in examples:
    print(f"TASK: {task}")
    print(build_instruction(task, content))
    print("-" * 60)

### Interpreting Example 5

The code does not call an LLM. It demonstrates the pattern used when working with one:

```text
Task Instruction + Task Data → Model Behaviour
```

The same model can receive different instructions and perform different transformations.

This is fundamentally different from a narrow classifier that can only return one of its trained labels.

### But generality introduces risk

A general model may:

- misunderstand instructions;
- produce an unexpected format;
- include unsupported claims;
- behave inconsistently;
- or attempt tasks beyond its capability.

Therefore, modern agents combine LLM flexibility with software controls.

## 17. From LLM to Agent

An LLM by itself receives input and generates output.

```text
Prompt → LLM → Text
```

An agent places the model inside a larger system.

```text
                         ┌──────────────┐
User Goal ──────────────▶│ Instructions │
                         └──────┬───────┘
                                ▼
                         ┌──────────────┐
                         │     LLM      │
                         │ decision     │
                         └──────┬───────┘
                                │
                     ┌──────────┴──────────┐
                     ▼                     ▼
                Text Response          Tool Call
                                           │
                                           ▼
                                     Environment
                                           │
                                           ▼
                                      Tool Result
                                           │
                                           └──────▶ LLM
```

The agent adds:

- instructions;
- tools;
- state;
- memory;
- control flow;
- validation;
- guardrails;
- and stopping conditions.

## 18. Model vs Agent

| Dimension | Language Model | AI Agent |
|---|---|---|
| Primary role | Generate or transform content | Pursue a goal through actions |
| Input | Prompt or context | Goal, context, observations, tool results |
| Output | Usually text or structured data | Responses, tool calls, handoffs, updates |
| External action | None by itself | Can act through tools |
| State | Limited to supplied context | May maintain explicit task state |
| Multi-step loop | Not inherently | Usually controlled by an execution loop |
| Permissions | Not applicable by itself | Must be explicitly bounded |
| Completion | Ends after generation | Ends when goal or stopping condition is reached |

A model may be the reasoning component of an agent, just as an engine is a component of a vehicle.

```text
Engine ≠ Vehicle
LLM ≠ Agent
```

## 19. Chatbot vs Assistant vs Workflow vs Agent

These terms are used inconsistently in industry. The following definitions are practical rather than universal.

### Chatbot

A chatbot primarily conducts a conversation.

```text
Message → Response
```

### Assistant

An assistant helps a user and may use tools, but usually remains user-directed.

```text
User Request → Suggestion or Tool-Assisted Response
```

### Workflow

A workflow executes a predefined sequence.

```text
Trigger → Step A → Step B → Step C
```

### Agent

An agent selects actions dynamically based on goals and observations.

```text
Goal → Observe → Choose Next Action → Act → Reassess
```

## 20. Detailed Comparison

| Feature | Chatbot | Assistant | Workflow | AI Agent |
|---|---|---|---|---|
| Main purpose | Conversation | User support | Process automation | Goal achievement |
| Path | Mostly conversational | User-guided | Predetermined | Dynamically selected |
| Tool use | Optional | Common | Defined in advance | Selected contextually |
| Memory | Often conversational | User/task context | Process state | Task state and memory |
| Planning | Minimal | Limited or user-led | Designed by developer | May generate or revise plans |
| Autonomy | Low | Low to medium | Low in reasoning, high in execution | Variable and bounded |
| Predictability | Medium | Medium | High | Lower without controls |
| Error recovery | Usually limited | Often asks user | Pre-programmed branches | May diagnose and re-plan |
| Best for | FAQs, dialogue | Productivity support | Stable repeatable process | Open-ended multi-step tasks |

A system can combine all four.

For example, a customer-support product may contain:

- a chatbot interface;
- an assistant that drafts answers;
- workflows for refunds;
- and an agent that investigates complex cases.

## 21. Fixed Workflow vs Agentic Workflow

Consider an invoice-processing system.

### Fixed workflow

```text
Receive invoice
      ↓
Extract fields
      ↓
Validate total
      ↓
Store record
      ↓
Send confirmation
```

The route is predefined.

### Agentic workflow

```text
Receive invoice
      ↓
Inspect document
      ↓
Is information missing?
  ├── Yes → Search email thread or ask supplier
  └── No
      ↓
Does total conflict with purchase order?
  ├── Yes → Investigate and request approval
  └── No
      ↓
Choose appropriate accounting action
```

The agentic version selects among possible actions based on observations.

### Important lesson

A workflow is not inferior to an agent.

Use a workflow when the process is known and stable.  
Use agentic decision-making where flexible interpretation or recovery is necessary.

In [ ]:
# Example 6: Fixed workflow and dynamic agent-like routing

def fixed_invoice_workflow(invoice: dict) -> list[str]:
    """Always executes the same sequence."""
    return [
        "Extract invoice fields",
        "Validate total",
        "Store invoice",
        "Send confirmation",
    ]


def dynamic_invoice_router(invoice: dict) -> list[str]:
    """Selects actions according to the observed invoice state."""
    actions = ["Extract invoice fields"]

    if not invoice.get("supplier"):
        actions.append("Request missing supplier information")
        return actions

    if invoice.get("total") != invoice.get("purchase_order_total"):
        actions.append("Investigate purchase-order mismatch")
        actions.append("Request human approval")
        return actions

    actions.append("Approve invoice")
    actions.append("Store invoice")
    return actions


invoices = [
    {
        "supplier": "Northstar Ltd",
        "total": 900,
        "purchase_order_total": 900
    },
    {
        "supplier": "",
        "total": 650,
        "purchase_order_total": 650
    },
    {
        "supplier": "Eastbridge Services",
        "total": 1_200,
        "purchase_order_total": 950
    },
]

for index, invoice in enumerate(invoices, start=1):
    print(f"Invoice {index}")
    print("Fixed:", fixed_invoice_workflow(invoice))
    print("Dynamic:", dynamic_invoice_router(invoice))
    print()

### Interpreting Example 6

The fixed workflow always returns the same sequence.

The dynamic router changes its next action based on:

- missing information;
- mismatched totals;
- and risk.

Although this Python function is still rule-based, it demonstrates the structural difference between:

```text
Always execute the next predefined step
```

and:

```text
Inspect the current state and select the next useful action
```

A modern LLM-powered agent can make this selection using language-based reasoning, tool results, and structured state.

## Knowledge Check 3

For each system, classify it as primarily a chatbot, assistant, workflow, or agent. More than one label may apply.

1. A website FAQ bot that answers from a fixed list.
2. A script that sends certificates after a form submission.
3. A coding tool that reads a repository, edits files, runs tests, and fixes failures.
4. A calendar assistant that suggests available meeting times.
5. A refund process with fixed eligibility rules.
6. A research system that searches, compares sources, changes queries, and writes a cited report.
7. A support bot that collects information and then invokes a fixed ticket workflow.

For each answer, identify:

- the goal;
- observations;
- actions;
- whether the path is fixed;
- and the level of autonomy.

## 22. Characteristics of a Modern AI Agent

A modern agent often combines the following characteristics:

```text
┌────────────────────────────────────────┐
│           MODERN AI AGENT              │
├────────────────────────────────────────┤
│ 1. Goal-oriented behaviour             │
│ 2. Natural-language understanding      │
│ 3. Context-sensitive decision-making   │
│ 4. Tool use                             │
│ 5. Multi-step execution                │
│ 6. State and memory                     │
│ 7. Planning and re-planning             │
│ 8. Feedback processing                 │
│ 9. Guardrails and permissions           │
│10. Clear stopping conditions            │
└────────────────────────────────────────┘
```

Not every agent needs every capability.

A one-step routing agent may not need long-term memory.  
A research agent may need tools and iterative planning.  
A financial agent may require strict approvals and deterministic validation.

## 23. Goal-Oriented Behaviour

A modern agent is usually given an outcome rather than a complete sequence of instructions.

### Procedure-based request

> Open file A, copy values from column B, calculate the total, and email it.

### Goal-based request

> Prepare and send the weekly expense summary.

The second request leaves decisions to the system:

- Which file contains the expenses?
- Which rows belong to the current week?
- How should missing values be handled?
- Who should receive the summary?
- Should the email be sent immediately or drafted for approval?

Greater freedom can improve usability, but it also increases responsibility and risk.

## 24. Context-Sensitive Decision-Making

The same user message can require different actions in different contexts.

Message:

> “Please cancel it.”

Possible meanings:

- cancel a meeting;
- cancel an order;
- cancel a subscription;
- cancel the current agent run.

The correct action depends on:

- conversation history;
- current task;
- user identity;
- permissions;
- relevant records;
- and timing.

```text
Message alone → Ambiguous
Message + Context → Actionable
```

Modern agents need context management because language is inherently dependent on prior information.

## 25. Tool Use

An LLM's internal knowledge is not enough for many tasks.

It may need tools to:

- search current information;
- query a database;
- calculate accurately;
- read a file;
- send an email;
- update a calendar;
- execute code;
- or interact with another system.

```text
LLM Knowledge
    +
External Tools
    =
Actionable Capability
```

Without tools, a model can describe how to book a flight.  
With tools, an authorised agent may search flights and prepare a booking.  
With approval and payment tools, it may complete the booking.

Tools transform an agent from a text generator into a system capable of affecting the environment.

## 26. Multi-Step Execution and Feedback

An agent rarely knows the final answer before interacting with the environment.

Example research task:

```text
Search for sources
      ↓
Inspect results
      ↓
Results insufficient?
  ├── Yes → Refine query and search again
  └── No
      ↓
Compare evidence
      ↓
Draft answer
      ↓
Check citations
      ↓
Return result
```

The result of one action becomes the observation for the next action.

```text
Action₁ → Observation₂ → Action₂ → Observation₃
```

This feedback loop is one of the defining properties of an agent.

## 27. State, Memory, and Continuity

An agent may need to remember:

- the user's goal;
- information already collected;
- actions already completed;
- tool results;
- failures;
- approvals;
- and remaining work.

Without state, the agent may:

- repeat actions;
- forget constraints;
- ask the same question;
- lose progress;
- or contradict earlier decisions.

A simple task state might look like:

```python
{
    "goal": "Prepare workshop schedule",
    "completed": ["collect availability"],
    "pending": ["select dates", "draft schedule"],
    "constraints": ["avoid weekends"],
    "approval_required": True
}
```

Part 2 will study context, state, and memory in greater depth.

In [ ]:
# Example 7: Tracking task state

task_state = {
    "goal": "Prepare workshop schedule",
    "completed": [],
    "pending": [
        "Collect instructor availability",
        "Select suitable dates",
        "Draft the schedule",
        "Request approval"
    ],
    "constraints": [
        "Avoid weekends",
        "Use 90-minute sessions"
    ]
}


def complete_next_step(state: dict) -> str:
    """Move the next pending task into the completed list."""
    if not state["pending"]:
        return "Task already complete."

    next_step = state["pending"].pop(0)
    state["completed"].append(next_step)
    return f"Completed: {next_step}"


print("Goal:", task_state["goal"])
print("Constraints:", task_state["constraints"])
print()

for _ in range(3):
    print(complete_next_step(task_state))
    print("Completed:", task_state["completed"])
    print("Pending:", task_state["pending"])
    print()

### Interpreting Example 7

The example is not intelligent by itself. It demonstrates why explicit state matters.

At any moment, the system can answer:

- What is the goal?
- What has been completed?
- What remains?
- Which constraints apply?

In production agents, state may be updated by:

- deterministic code;
- tool outputs;
- user input;
- model decisions;
- or a combination of all four.

> **Engineering principle**
>
> Important task facts should not exist only in unstructured model conversation. Store critical state explicitly whenever possible.

## 28. Guardrails and Bounded Autonomy

A capable agent should not automatically be allowed to perform every possible action.

Examples of actions that may require approval:

- spending money;
- deleting records;
- sending external emails;
- publishing content;
- modifying production systems;
- sharing confidential information.

A useful permission model is:

```text
Low risk:
Agent may act automatically

Medium risk:
Agent may act within defined limits

High risk:
Agent must request human approval

Prohibited:
Agent must never act
```

Modern agent design is not only about increasing autonomy. It is about selecting the **right level of autonomy**.

## 29. Real-World Case Study: Research Agent

Goal:

> “Produce a concise report comparing three approaches to reducing urban traffic congestion.”

Possible execution:

```text
1. Interpret the research question
2. Break it into sub-questions
3. Search for credible sources
4. Read relevant documents
5. Record evidence
6. Compare approaches
7. Identify disagreements
8. Draft the report
9. Verify citations
10. Return the final response
```

### Agent characteristics present

| Characteristic | Example |
|---|---|
| Goal orientation | Produce the comparison report |
| Tool use | Web search and document reading |
| Planning | Divide the topic into sub-questions |
| Feedback | Refine searches when evidence is weak |
| State | Track sources and findings |
| Guardrails | Cite evidence; avoid unsupported claims |
| Stopping condition | Report meets scope and citation requirements |

This is more than a chatbot because the system performs a structured sequence of actions.

## 30. Real-World Case Study: Coding Agent

Goal:

> “Add input validation to the registration service and ensure all tests pass.”

Possible actions:

1. inspect the repository;
2. locate the relevant service;
3. read existing tests;
4. modify code;
5. run the test suite;
6. inspect failures;
7. revise the implementation;
8. summarise changes.

```text
Read Code
   ↓
Plan Change
   ↓
Edit Files
   ↓
Run Tests
   ↓
Tests Pass?
 ├── No → Diagnose and revise
 └── Yes → Report completion
```

The test result acts as feedback.

A coding agent therefore needs more than code generation. It needs:

- repository tools;
- file-editing tools;
- execution tools;
- state;
- error recovery;
- and limits on what it may modify.

## 31. When Not to Use an Agent

Agents are not always the best solution.

Prefer deterministic software when:

- the process is stable;
- the rules are precise;
- the same path should always run;
- auditability is critical;
- latency must be minimal;
- errors are unacceptable;
- language interpretation is unnecessary.

Examples:

- tax calculations;
- database constraints;
- payment totals;
- access-control checks;
- cryptographic operations;
- safety interlocks.

A strong production architecture often looks like:

```text
LLM decides what needs to happen
        ↓
Deterministic software validates whether it may happen
        ↓
Tool executes within explicit limits
```

This combines flexibility with control.

## 32. Common Misconceptions

### Misconception 1: Every chatbot is an agent

A chatbot may only generate responses. It becomes more agent-like when it selects actions, uses tools, maintains state, and pursues goals.

### Misconception 2: Every workflow is an agent

A workflow may execute a fixed sequence without dynamic decision-making.

### Misconception 3: More autonomy is always better

More autonomy also creates more risk, unpredictability, and monitoring requirements.

### Misconception 4: The LLM should perform every task

Calculations, validation, access control, and transactions are often better handled by deterministic tools.

### Misconception 5: Tools automatically make a system agentic

A model that always calls one fixed tool may still be a tool-assisted application rather than a meaningful agent.

### Misconception 6: Agents eliminate software engineering

Agents increase the importance of architecture, testing, security, observability, and evaluation.

## Knowledge Check 4

1. List six characteristics of a modern AI agent.
2. Why is tool use important?
3. How does feedback influence the next action?
4. What is explicit task state?
5. Why should important state not exist only in conversation text?
6. Give three actions that should require human approval.
7. Why is a fixed workflow sometimes better than an agent?
8. What combination of LLM and deterministic software would you use for a payment assistant?
9. Explain the difference between capability and permission.
10. Is a chatbot with one calculator tool necessarily an agent? Defend your answer.

## 33. Mini Lab — Design a University Admission Agent

Design an agent that helps students understand and complete university admissions.

### Step 1: Define the goal

Write one clear goal statement.

Example format:

> Help an applicant identify suitable programmes and complete an accurate application without violating admissions policy.

### Step 2: Define observations

What information can the agent receive?

- student message;
- programme catalogue;
- eligibility rules;
- uploaded documents;
- application status.

### Step 3: Define tools

Possible tools:

- programme search;
- eligibility checker;
- document reader;
- deadline lookup;
- email drafting;
- application-status API.

### Step 4: Define boundaries

Which actions may be automatic?  
Which require approval?  
Which are prohibited?

### Step 5: Define stopping conditions

When has the agent completed the task?

> **Classroom activity**
>
> Work in groups of three. One student represents the user, one the agent designer, and one the safety reviewer.

## 34. Design Worksheet

Complete the following table.

| Design Question | Your Answer |
|---|---|
| Who is the user? | |
| What is the user's goal? | |
| What environment does the agent operate in? | |
| What observations can it receive? | |
| What actions can it perform? | |
| Which tools are required? | |
| What state must it retain? | |
| What can go wrong? | |
| Which actions need approval? | |
| How will success be measured? | |
| When should the agent stop? | |

### Extension

Identify which components should be:

- LLM-based;
- deterministic;
- human-controlled.

## 35. Exercises

### Exercise 1 — Conceptual Comparison

Explain the differences among:

- symbolic AI;
- machine learning;
- deep learning;
- foundation models;
- LLMs;
- modern AI agents.

### Exercise 2 — Rule-Based Limitation

Create a keyword-based classifier for five student-support intents. Test it using alternative expressions that do not contain the expected keywords. Document the failures.

### Exercise 3 — System Classification

For each system below, classify it as a chatbot, assistant, workflow, agent, or hybrid:

- FAQ bot;
- automated payroll process;
- research assistant;
- calendar scheduler;
- autonomous coding system.

### Exercise 4 — Agent Decomposition

Choose one real-world goal and identify:

- observations;
- actions;
- tools;
- state;
- constraints;
- feedback;
- stopping condition.

### Exercise 5 — Hybrid Architecture

Design a system in which an LLM interprets an expense request, but deterministic code validates policy and calculates reimbursement.

## 36. End-of-Part Quiz

1. Which approach primarily depends on explicitly encoded symbols and rules?  
   A. Deep learning  
   B. Symbolic AI  
   C. Foundation modelling  
   D. Reinforcement learning

2. What is the defining change introduced by machine learning?  
   A. More user interfaces  
   B. Patterns are learned from data  
   C. All rules disappear  
   D. Models become agents

3. A foundation model is best described as:  
   A. A model designed for one fixed task  
   B. A broad model adaptable to many tasks  
   C. A database of rules  
   D. A workflow engine

4. An LLM by itself is not necessarily an agent because it may lack:  
   A. Tokens  
   B. Parameters  
   C. tools and an execution loop  
   D. text generation

5. A workflow differs from an agent because a workflow usually:  
   A. has no goal  
   B. follows a predefined path  
   C. cannot use software tools  
   D. requires an LLM

6. Which is the clearest example of feedback?  
   A. Reading the original instruction  
   B. Receiving a failed test result after editing code  
   C. Defining the goal  
   D. Choosing a model name

7. Explicit task state helps prevent:  
   A. all model errors  
   B. repeated or forgotten work  
   C. every security issue  
   D. the need for tools

8. Which action most clearly requires human approval?  
   A. Formatting a date  
   B. Reading a public webpage  
   C. Transferring money  
   D. Counting words

9. When is a deterministic workflow usually preferable?  
   A. When the task is stable and precisely defined  
   B. When the goal is ambiguous  
   C. When the environment is unknown  
   D. When the next action cannot be predicted

10. The strongest production design commonly combines:  
    A. LLM flexibility with deterministic validation  
    B. only free-form language generation  
    C. unlimited agent permissions  
    D. no explicit state

<details>
<summary><strong>Answer key</strong></summary>

1-B, 2-B, 3-B, 4-C, 5-B, 6-B, 7-B, 8-C, 9-A, 10-A

</details>

## 37. Summary

This notebook explained why modern AI agents emerged.

### Key ideas

- Traditional software is strong when rules are precise and stable.
- Rule-based systems struggle with flexible language and combinatorial complexity.
- Symbolic AI represents knowledge explicitly.
- Expert systems encode specialist rules.
- Machine learning learns patterns from examples.
- Deep learning learns layered representations.
- Foundation models support many downstream tasks.
- LLMs provide a general natural-language interface.
- An LLM is a component, not automatically an agent.
- A modern agent combines a model with goals, tools, state, feedback, controls, and stopping conditions.
- Chatbots, assistants, workflows, and agents represent different interaction and control patterns.
- Workflows remain preferable for stable, deterministic processes.
- Reliable production systems combine LLM flexibility with deterministic validation and human oversight.

```text
Rules
  ↓
Learned Models
  ↓
Foundation Models
  ↓
Language Models
  ↓
Language Model + Tools + State + Loop + Guardrails
  ↓
Modern AI Agent
```

## 38. Preview of Notebook 1.2 — Part 2

Part 2 will open the modern agent and study its internal components.

Expected topics:

1. anatomy of an LLM-powered agent;
2. instructions and role definition;
3. model selection;
4. tools and function calling;
5. context windows;
6. state and memory;
7. planning and task decomposition;
8. agent execution loops;
9. structured outputs;
10. human-in-the-loop control.

The central question will be:

> How does a modern agent decide what to do next, use a tool, inspect the result, and continue until the task is complete?

---

**End of Notebook 1.2 — Part 1**